# DDOI positions — PROXY_trade_flow_DDOI

**Paper:** Dealer Directional OI (PDF p. 3) · **NOTES:** [`NOTES.md`](NOTES.md) · **EXP:** [`EXP_REPORT.md`](EXP_REPORT.md)  
**Artifacts:** `out/ddoi_positions/` · **Lib:** `research.lib.squeeze`

Paper DDOI uses trade direction **and** verified ΔOI. Desk warehouse `open_interest` is **futures-only** → option inventory is **PROXY_trade_flow_DDOI** (aggressor × qty by instrument). Never treat futures OI as option DDOI. Decision: **Hold**.


In [ ]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

BOOK = Path(os.environ.get("ARES_MICROSTRUCTURE") or (Path.home() / "srv" / "ares-microstructure")) / "research" / "books" / "squeeze_metrics"
OUT = BOOK / "out"
SCRIPTS = BOOK / "scripts"
ROOT = BOOK.parents[2]
sys.path.insert(0, str(SCRIPTS))
sys.path.insert(0, str(ROOT))
from _nb_common import (
    load_json, gex_day_table, safe_corr, partial_corr_vs_rv, lead_lag_day, SIGNAL_BOARD,
)
from certified_panel import load_certified, primary_days, spot_l2_days, gex_panel_days, DDOI_LABEL, GEX_PROXY_LABEL
from research.lib import squeeze  # noqa: E402

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

def show_pngs(fig_dir: Path, pattern="*.png"):
    if not fig_dir.is_dir():
        print("no figs dir", fig_dir)
        return
    for p in sorted(fig_dir.glob(pattern)):
        display(Markdown(f"**{p.name}** — `{p.relative_to(BOOK)}`"))
        display(Image(filename=str(p)))

def print_board(rows=None):
    rows = rows or SIGNAL_BOARD
    print(f"{'id':34s} {'gate':6s} note")
    print("-" * 88)
    for i, g, n in rows:
        print(f"{i:34s} {g:6s} {n}")

CERT = load_certified()
PRIMARY = primary_days(CERT)
SPOT = spot_l2_days(CERT)
GEX_DAYS = gex_panel_days(CERT)
print("BOOK", BOOK)
print("certified primary n=", len(PRIMARY), PRIMARY)
print("panel_gex_options n=", len(GEX_DAYS), GEX_DAYS)
print("spot_l2 subpanel n=", len(SPOT), SPOT)
print("DDOI label", DDOI_LABEL, "| GEX proxy", GEX_PROXY_LABEL)
print("trade_synth QUARANTINED — never primary; warehouse OI futures-only unused")


In [ ]:

summary = load_json(OUT / "ddoi_positions" / "summary.json")
rows = (summary or {}).get("rows") or []
df = pd.DataFrame(rows)
print(json.dumps({k: (summary or {}).get(k) for k in ("n_ok","days_ok","mean_abs_ddoi","label","decision")}, indent=2, default=str))
display(df[["day","ok","n_trades","n_instruments","ddoi_sum","ddoi_abs_sum","ddoi_n_nonzero","label"]].round(4) if len(df) and "n_trades" in df.columns else df)


## Coverage — trades / instruments / |DDOI|


In [ ]:

ok = df[df["ok"] == True] if len(df) and "ok" in df.columns else df
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6))
if len(ok):
    axes[0].bar(ok["day"], ok["n_trades"], color="#3d5a80")
    axes[0].tick_params(axis="x", rotation=70); axes[0].set_title("Option trades / day")
    axes[1].bar(ok["day"], ok["n_instruments"], color="#98c1d9")
    axes[1].tick_params(axis="x", rotation=70); axes[1].set_title("Instruments with flow")
    axes[2].bar(ok["day"], ok["ddoi_abs_sum"], color="#ee6c4d")
    axes[2].tick_params(axis="x", rotation=70); axes[2].set_title("|DDOI| sum (PROXY)")
else:
    for ax in axes:
        ax.text(0.5, 0.5, "run exp_ddoi_panel.py", ha="center", transform=ax.transAxes)
plt.tight_layout(); plt.show()
print("LABEL", DDOI_LABEL, "| futures OI QUARANTINED")


## Top instruments (latest ok day)


In [ ]:

if len(ok):
    last = ok.iloc[-1]
    day = last["day"]
    day_rec = load_json(OUT / "ddoi_positions" / f"day_{day}.json") or {}
    top = pd.DataFrame(day_rec.get("top_instruments") or [], columns=["instrument_id","ddoi"])
    display(top.head(15))
    print("honesty:", day_rec.get("honesty"))
else:
    print("no ok days")


## Signal board


In [ ]:

print_board([
    ("info.ddoi_coverage", "Hold", f"{DDOI_LABEL}; futures OI unused; n_ok={(summary or {}).get('n_ok')}"),
    ("risk.gex_exposure", "Hold", "Downstream GEX uses this PROXY — not paper DDOI"),
    ("data.trade_synth", "Kill", "never SoT"),
    ("alpha.tob_cross_arb", "Kill", "never Promote"),
])
